In [1]:
# rag_chroma_with_caches.py
# RAG (LangChain + Chroma) with:
#   1) Embedding cache → CacheBackedEmbeddings + LocalFileStore
#   2) LLM answer cache → langchain.llm_cache via SQLiteCache
#
# Usage:
#   pip install -U langchain langchain-openai chromadb pypdf python-dotenv
#   export OPENAI_API_KEY=sk-...   # (Windows: setx OPENAI_API_KEY "sk-...")
#   python rag_chroma_with_caches.py
#
# Optional: put PDFs/TXTs under ./data to index; otherwise sample texts are used.

import os, time
from pathlib import Path
from dotenv import load_dotenv

# LangChain core + OpenAI wrappers
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
#from langchain.chat_models import ChatOpenAI
#from langchain.embeddings import OpenAIEmbeddings
#from langchain.vectorstores import Chroma
#from langchain.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader,
    PyPDFLoader,
)
from langchain_community.vectorstores import Chroma


#from langchain_community.vectorstores import Chroma
#from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
#from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# New-style runnables & prompt
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Caches
import langchain
#from langchain.cache import SQLiteCache
from langchain_community.cache import SQLiteCache
#from langchain.storage import LocalFileStore
#from langchain_core.stores import LocalFileStore
#from langchain_community.storage import LocalFileStore
#from langchain_text_splitters import LocalFileStore
#from langchain.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore
from langchain_classic.embeddings import CacheBackedEmbeddings
load_dotenv()

# ---------------------------
# Config
# ---------------------------
PERSIST_DIR     = "./chroma_db1"                       # Chroma persistence
EMB_CACHE_DIR   = "./emb_cache1"                       # Disk cache for embeddings
LLM_CACHE_PATH  = "./.langchain_llm_cache1sqlite"     # SQLite cache for LLM outputs
DATA_DIR        = "./data"                            # PDFs/TXTs go here
COLLECTION      = "docs1"
EMBED_MODEL     = "text-embedding-3-small"            # cheap & good
CHAT_MODEL      = "gpt-4o-mini"                       # any OpenAI chat model

# Turn on LLM answer cache (affects llm.invoke inside chains too)
Path(LLM_CACHE_PATH).parent.mkdir(parents=True, exist_ok=True)
langchain.llm_cache = SQLiteCache(database_path=LLM_CACHE_PATH)

# ---------------------------
# Helpers
# ---------------------------
def load_docs():
    """Load docs from ./data (PDF/TXT). Fall back to small samples."""
    docs = []
    data_path = Path(DATA_DIR)
    print(data_path)
    if data_path.exists():
        # pass 1: text-like files (DirectoryLoader + TextLoader)
        loader = DirectoryLoader(
            DATA_DIR,
            glob="**/*",
            loader_cls=TextLoader,
            show_progress=True,
            use_multithreading=True,
        )
        try:
            docs.extend(loader.load())
        except Exception:
            pass
        # pass 2: PDFs (PyPDFLoader)
        for pdf in data_path.rglob("*.pdf"):
            try:
                docs.extend(PyPDFLoader(str(pdf)).load())
            except Exception as e1:
                print(e1)
                pass
    print("!!!!!!!!!!!!!!!!!!!")
    print(docs)
    print("(((((((((((((())))))))))))))")

    if not docs:
        from langchain.schema import Document
        docs = [
            Document(page_content=("LangChain is a framework for developing LLM apps. "
                                   "It integrates vector stores like Chroma and supports RAG pipelines."),
                     metadata={"source": "sample:langchain"}),
            Document(page_content=("Chroma is an open-source embedding DB (vector store) that stores "
                                   "document embeddings and enables similarity search."),
                     metadata={"source": "sample:chroma"}),
        ]
    print(docs)
    return docs


def build_or_load_chroma(cached_embeddings) -> Chroma:
    """Create/load a persistent Chroma index using cached embeddings."""
    Path(PERSIST_DIR).mkdir(parents=True, exist_ok=True)
    # If an existing index is present, load it
    index_exists = (Path(PERSIST_DIR) / "chroma.sqlite3").exists()
    if index_exists:
        return Chroma(
            collection_name=COLLECTION,
            embedding_function=cached_embeddings,
            persist_directory=PERSIST_DIR,
        )

    # Otherwise, index docs
    docs = load_docs()
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
    chunks = splitter.split_documents(docs)

    vs = Chroma.from_documents(
        documents=chunks,
        embedding=cached_embeddings,
        collection_name=COLLECTION,
        persist_directory=PERSIST_DIR,
    )
    vs.persist()
    return vs


def make_rag_chain(retriever, llm):
    """Simple RAG chain: retrieve → prompt → LLM → string."""
    def format_docs(docs):
        return "\n\n".join(
            f"Source: {d.metadata.get('source','?')}\n{d.page_content}" for d in docs
        )

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system",
             "You are a concise, helpful assistant. Use the provided context to answer. "
             "If the answer isn't in the context, say so.\n\nContext:\n{context}"),
            ("human", "{question}"),
        ]
    )

    chain = (
        {
            "context": retriever | (lambda docs: format_docs(docs)),
            "question": RunnablePassthrough(),
        }
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain


def timed(fn):
    def _inner(*args, **kwargs):
        t0 = time.time()
        out = fn(*args, **kwargs)
        return out, time.time() - t0
    return _inner

import pandas as pd
# ---------------------------
# Main
# ---------------------------
if __name__ == "__main__":
    # 1) Embedding cache: stores content-hash → vector on disk
    Path(EMB_CACHE_DIR).mkdir(parents=True, exist_ok=True)
    base_embeddings = OpenAIEmbeddings(model=EMBED_MODEL)
    byte_store = LocalFileStore(EMB_CACHE_DIR)
    cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
        base_embeddings,
        byte_store,
        namespace=f"{EMBED_MODEL}-v1",   # separate namespace per settings/model
    )

    # 2) Persistent Chroma index (load if exists; build if not)
    vstore = build_or_load_chroma(cached_embeddings)
    retriever = vstore.as_retriever(search_kwargs={"k": 4})

    # 3) LLM (benefits from SQLite LLM cache set above)
    llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)

    # 4) RAG chain
    chain = make_rag_chain(retriever, llm)
    df = pd.read_csv(r"C:\Users\surya.adatravu\Documents\RAGAnalysis\RA_FSM_QA.csv")
    df["t0"]=0
    df["t1"]=0
    df["ans0"]=''
    df['ans1']=''
    
    for i1 in range(0,df.shape[0]):
        question = df.loc[i1,"Question"]

        # 5) Demo: ask the same question twice to see cache effects
        #question = "What is Chroma and how does LangChain use it in a RAG pipeline?"

        ans1, t1 = timed(chain.invoke)(question)
        df.loc[i1,'t0']=t1
        df.loc[i1,'ans0']=ans1
#         print("\n--- First answer (warming caches) ---")
#         print(ans1)
#         print(f"(took {t1:.2f}s)")

        ans2, t2 = timed(chain.invoke)(question)
        df.loc[i1,'t1']=t2
        df.loc[i1,'ans1']=ans2
#         print("\n--- Second answer (LLM answer served from cache) ---")
#         print(ans2)
#         print(f"(took {t2:.2f}s)")
        #break
    df.to_csv("results_b_r1.csv")

    print("\nCache locations:")
    print(f"• Chroma DB:        {Path(PERSIST_DIR).resolve()}")
    print(f"• Embedding cache:  {Path(EMB_CACHE_DIR).resolve()}")
    print(f"• LLM cache (SQL):  {Path(LLM_CACHE_PATH).resolve()}")


C:\Users\surya.adatravu\AppData\Local\anaconda3\envs\r1\Lib\site-packages\langchain_classic\embeddings\cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()
C:\Users\surya.adatravu\AppData\Local\Temp\ipykernel_50280\4253702193.py:121: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  retu


Cache locations:
• Chroma DB:        C:\Users\surya.adatravu\Documents\RAGAnalysis\chroma_db1
• Embedding cache:  C:\Users\surya.adatravu\Documents\RAGAnalysis\emb_cache1
• LLM cache (SQL):  C:\Users\surya.adatravu\Documents\RAGAnalysis\.langchain_llm_cache1sqlite


In [3]:
pip show langchain

Name: langchain
Version: 1.0.3
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: C:\Users\surya.adatravu\AppData\Local\anaconda3\envs\r1\Lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [10]:
pip install langchain-storage

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement langchain-storage (from versions: none)
ERROR: No matching distribution found for langchain-storage


In [5]:
pip install chromadb

  Using cached build-1.3.0-py3-none-any.whl.metadata (5.6 kB)
  Using cached uvicorn-0.38.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached opentelemetry_api-1.38.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.38.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached opentelemetry_sdk-1.38.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached tokenizers-0.22.1-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
  Using cached PyPika-0.48.9.tar.gz (67 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached importlib_resources-6.5.2-py3-none-any.whl.metadata (3.9 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-win_amd64.whl.metadata (10 kB)
  Using cached kubernetes-34.1.0-